# BỐI CẢNH PHÂN TÍCH - GIAI ĐOẠN TĂNG TRƯỞNG


## I. TỪ BÀI TOÁN QUẢN TRỊ ĐẾN CÂU HỎI PHÂN TÍCH

Mọi nỗ lực từ bán hàng, giao hàng đến tối ưu nguyên liệu rốt cuộc cũng phải quy về một câu hỏi cuối cùng: **“Danny thực sự bỏ túi được bao nhiêu tiền?”** Ở giai đoạn tăng trưởng, tư duy “sinh tồn” không còn phù hợp. Thay vào đó, Danny cần một **tầm nhìn tài chính rõ ràng**: doanh thu đến từ đâu, chi phí chảy đi đâu, và lợi nhuận ròng còn lại là bao nhiêu sau khi trả hết các khoản phí vận hành – bao gồm cả tiền công cho những người giao hàng đã đồng hành cùng anh từ những ngày đầu.

Song song với bài toán tiền bạc, Danny bắt đầu nghĩ đến **tương lai dài hạn**. Làm sao để khách hàng không chỉ quay lại, mà còn trở thành “đại sứ thương hiệu” cho Pizza Runner? Một hệ thống đánh giá sao cho runner là lời giải. Nó vừa là công cụ kiểm soát chất lượng, vừa là kênh lắng nghe phản hồi quý giá. Nhưng thiết kế nó ra sao? Thu thập dữ liệu gì, và kết nối với các bảng hiện có thế nào để tạo nên một bức tranh toàn cảnh về hiệu suất?

Đây chính là lúc Danny **“bứt phá”**: từ một quán pizza nhỏ, anh bắt đầu vận hành như một doanh nghiệp thực thụ – có kế toán, có định giá, có chăm sóc khách hàng. Mọi quyết định tăng giá, giảm giá, thưởng cho runner hay mở thêm chi nhánh đều phải dựa trên dữ liệu, chứ không còn là cảm tính.


| Câu hỏi quản trị | Các câu hỏi phân tích |
|------------------|-----------------------|
| Giá bán hiện tại có tối ưu chưa? | D.1 – Doanh thu nếu giá cố định 12$ |
| Có nên thu phí topping thêm? | D.2 – Doanh thu tăng thêm khi thu phí topping thêm 1$ |
| Khách hàng có hài lòng với dịch vụ giao hàng không? | D.3 – Thiết kế bảng đánh giá runner (1-5 sao) <br> D.4 – Bảng tổng hợp đơn hàng – giao hàng – đánh giá |
| Sau khi trả phí tài xế, Danny còn lãi bao nhiêu? | D.5 – Lợi nhuận ròng sau khi trả phí runner 0.30$/km |



## II. PHÂN TÍCH VÀ ĐỀ XUẤT HÀNH ĐỘNG

In [ ]:
%sql
USE pizza_runner;

### NHÓM 1: ĐÁNH GIÁ HIỆU QUẢ MÔ HÌNH ĐỊNH GIÁ HIỆN TẠI

#### D.1. If a Meat Lovers pizza costs $12 and Vegetarian costs $10 and there were no charges for changes - how much money has Pizza Runner made so far if there are no delivery fees?


In [ ]:
%sql
WITH RECURSIVE money_each_pizza_in_delivered_orders AS(
		SELECT
				co.order_id,
				co.pizza_id,
				pn.pizza_name,
				co.customer_orders_id,
				CASE
					WHEN co.pizza_id = 1 THEN 12
					ELSE 10
				END AS money_for_pizza
		FROM destination.customer_orders co
		JOIN destination.pizza_names pn ON co.pizza_id = pn.pizza_id
		WHERE co.order_id NOT IN (SELECT ro.order_id FROM destination.runner_orders ro WHERE ro.cancellation IS NOT NULL)
		GROUP BY co.order_id,
				 co.pizza_id,
				 pn.pizza_name,
				 co.customer_orders_id
--		ORDER BY co.order_id ASC,
--				 co.pizza_id ASC,
--				 co.customer_orders_id ASC
), 	 total_money_for_specific_pizza  AS(
SELECT
		t.pizza_name,
		SUM(t.money_for_pizza) AS total_money_for_specific_pizza
FROM money_each_pizza_in_delivered_orders t
GROUP BY t.pizza_name
)
SELECT
		SUM(total_money_for_specific_pizza) AS total_money
FROM total_money_for_specific_pizza t

total_money
138


**Đọc dữ liệu**

  * Với giá cố định Meatlovers = 12 đô la, Vegetarian = 10 đô la, không phí extras và không phí giao hàng, tổng doanh thu từ các đơn giao thành công là 138 đô la.

**Insights**
  
  * Doanh thu hiện tại còn khiêm tốn (138 đô la cho toàn bộ các đơn thành công), phù hợp với giai đoạn đầu (early stage).
  * Meatlovers đóng góp chính vào doanh thu nhờ giá cao hơn và được đặt nhiều hơn (10 pizza so với 4 pizza của Vegetarian).

**Đề xuất hành động**
  
  * Giữ giá cơ bản hiện tại: Duy trì mức giá 12 đô la cho Meatlovers và 10 đô la cho Vegetarian làm nền tảng, vì chúng đã phản ánh đúng nhu cầu thị trường (Meatlowers chiếm ưu thế).
  * Theo dõi doanh thu theo loại pizza: Thiết lập bảng điều khiển (dashboard) để giám sát doanh thu từng loại pizza theo tuần. Dữ liệu hiện tại cho thấy Meatlovers tạo ra doanh thu lớn hơn, do đó cần ưu tiên marketing cho sản phẩm này. Nếu chi phí nguyên liệu (từ File 03) tăng, có thể điều chỉnh giá bán lẻ phù hợp.


### NHÓM 2: ĐO LƯỜNG TIỀM NĂNG TỪ PHÍ TOPPING THÊM

#### D.2. What if there was an additional 1 dollar charge for any pizza extras?
* **Add cheese is 1 dollar extra**

In [ ]:
%sql
;WITH RECURSIVE pre_data AS (
		SELECT
				v.order_id,
				v.pizza_id,
				v.customer_orders_id,
				v.total_topping_id,
				v.topping_id_for_specific_pizza,
				v.extras_topping_id_for_specific_pizza ,
				CASE
					WHEN v.extras_topping_id_for_specific_pizza IS NULL THEN 0
					ELSE 1
				END AS extra_money_for_extra_topping
		FROM destination.vw_pizza_ingredient_matrix v
--		ORDER BY
--				v.order_id,
--				v.pizza_id,
--				v.customer_orders_id,
--				v.total_topping_id,
--				v.topping_id_for_specific_pizza
)
, pre_calculate AS (
		SELECT
				p.order_id,
				 p.customer_orders_id,
				 p.pizza_id,
				 CASE
					WHEN p.pizza_id = 1 THEN 12
					ELSE 10
				END AS money_for_pizza,
				SUM(p.extra_money_for_extra_topping) AS extra_money_for_extra_topping
		FROM	pre_data p
		WHERE	p.order_id NOT IN (SELECT ro.order_id FROM destination.runner_orders ro WHERE ro.cancellation IS NOT NULL)
		GROUP BY p.order_id,
				 p.customer_orders_id,
				 p.pizza_id
--		ORDER BY
--				 p.order_id,
--				 p.customer_orders_id,
--				 p.pizza_id
)
SELECT
	  SUM(p.money_for_pizza + p.extra_money_for_extra_topping) AS total_money
FROM pre_calculate p

total_money
142


**Đọc dữ liệu**

  * Nếu thu thêm 1 đô la cho mỗi topping extra (ví dụ thêm Cheese, Bacon, v.v.), tổng doanh thu tăng lên 142 đô la, cao hơn 4 đô la so với mô hình không thu phí extras.

**Insights**
  
  * Phí extras mang lại thêm doanh thu dù số lượng thay đổi không nhiều, chủ yếu đến từ Bacon (topping được thêm nhiều nhất trong C.1 của File 03).
  * Tiềm năng tăng giá trị đơn hàng trung bình (AOV) là rõ rệt nếu triển khai thu phí topping một cách hợp lý.

**Đề xuất hành động**
  
  * **Áp dụng phí extras 1 đô la mỗi topping:** Triển khai ngay trên app hoặc website, đặc biệt với Bacon (topping được thêm nhiều nhất). Dữ liệu cho thấy chỉ cần 4 lần extra (Bacon) đã tạo thêm 4 đô la, tương ứng tỷ lệ tăng doanh thu khoảng 2.9% (từ 138 lên 142). Nếu khách hàng tiếp tục thêm nhiều hơn, doanh thu sẽ tăng đáng kể.
  * **Tạo gói combo extras phổ biến:** Ví dụ "Extra Bacon + Cheese" với giá 1.5 đô la thay vì 2 đô la, khuyến khích khách chọn nhiều topping hơn. Dữ liệu từ File 03 cho thấy Bacon và Cheese đều được ưa chuộng (Bacon được thêm, Cheese vừa được dùng nhiều vừa bị loại - nhưng với phí extra, khách có thể chủ động thêm nếu muốn).
  * **Mục tiêu:** Tăng doanh thu thêm 10-15% từ extras trong vòng 2 tháng tới.

### NHÓM 3: XÂY DỰNG HỆ THỐNG PHẢN HỒI VÀ ĐÁNH GIÁ CHẤT LƯỢNG

#### D.3.

**The Pizza Runner team now wants to add an additional ratings system that allows customers to rate their runner**

**how would you design an additional table for this new dataset**

**generate a schema for this new table and insert your own data for ratings for each successful customer order between 1 to 5.**

In [ ]:
%sql
CREATE TABLE destination.runner_ratings (
		rating_id		BIGINT GENERATED ALWAYS AS IDENTITY,
		order_id		INT NOT NULL,
		rating			INT NOT NULL, -- Rating 1-5 (enforced at application level)
		comment			STRING,
		-- Timestamp will be provided by application during INSERT
		rating_time		TIMESTAMP
)

In [ ]:
%sql
-- 2 Tạo dữ liệu giả
INSERT INTO destination.runner_ratings (order_id, rating, comment)
VALUES
    (1, 5, 'Giao hàng siêu nhanh, bánh còn nóng hổi!'),
    (2, 4, 'Anh shipper thân thiện, lịch sự.'),
    (3, 3, 'Giao hơi muộn một chút nhưng thái độ tốt.'),
    (4, 1, 'Bánh đến nơi bị xô lệch, shipper không cẩn thận!'),
    (5, 5, 'Tuyệt vời, không có gì để chê.'),
    (7, 4, NULL), -- Khách lười không viết bình luận, hệ thống vẫn chấp nhận
    (8, 5, 'Tài xế đi xe rất cẩn thận.'),
    (10, 5, 'Dịch vụ xuất sắc!');

num_affected_rows,num_inserted_rows
8,8


In [ ]:
%sql
SELECT *
FROM destination.runner_ratings;

rating_id,order_id,rating,comment,rating_time
1,1,5,"Giao hàng siêu nhanh, bánh còn nóng hổi!",null
2,2,4,"Anh shipper thân thiện, lịch sự.",null
3,3,3,Giao hơi muộn một chút nhưng thái độ tốt.,null
4,4,1,"Bánh đến nơi bị xô lệch, shipper không cẩn thận!",null
5,5,5,"Tuyệt vời, không có gì để chê.",null
6,7,4,null,null
7,8,5,Tài xế đi xe rất cẩn thận.,null
8,10,5,Dịch vụ xuất sắc!,null


#### D.4.

**Using your newly generated table - can you join all of the information together to form a table which has the following information for successful deliveries?**

* **customer_id**
* **order_id**
* **runner_id**
* **rating**
* **order_time**
* **pickup_time**
* **Time between order and pickup**
* **Delivery duration**
* **Average speed**
* **Total number of pizzas**

In [ ]:
%sql
CREATE VIEW destination.vw_efficiency_for_each_order AS
WITH RECURSIVE pre_data AS (
		SELECT
				-- Nhóm đơn hàng
				o.order_id,
				o.customer_id,
				o.order_time,

				-- Nhóm giao vận
				ro.runner_order_id,
				ro.runner_id,
				ro.pickup_time,
				ro.distance,
				ro.duration,
				ro.cancellation,

				-- Nhóm chất lượng dịch vụ
				rr.rating_id,
				rr.rating,
				rr.comment,
				rr.rating_time
		-- orders có order_id từ 1 đến 10
		FROM  destination.orders o
		-- runner_orders ro có order_id từ 1 đến 10 NÊN dùng JOIN vẫn còn order_id từ 1 -> 10
		JOIN destination.runner_orders ro	ON o.order_id = ro.order_id
		-- runner_ratings thiếu order_id = 6 và 9 cũng là những đơn hàng bị hủy và không giao , nên muốn giữ từ 1-> 10 thì phải LEFT JOIN
		LEFT JOIN destination.runner_ratings rr ON ro.order_id = rr.order_id
-- ORDER BY o.order_id ASC
),
count_number_of_pizza_in_each_order AS (
		SELECT
				co.order_id ,
				COUNT(co.customer_orders_id) AS number_of_pizzas
		FROM destination.customer_orders co
		GROUP BY co.order_id
)
SELECT
		p.customer_id,
		p.order_id,
		c.number_of_pizzas,
		p.runner_id,
		p.rating,
		p.order_time,
		p.pickup_time,
		p.distance,

		-- Thời gian từ lúc đặt hàng order_time đến lúc shipper nhận hàng đi giao (phút)
		DATEDIFF(MINUTE, p.order_time, p.pickup_time) AS preparation_time,
		-- Thời gian giao hàng
		p.duration,
		-- Tốc độ trung bình của runner
		-- v = s/t = s860/ t (km/h)
		ROUND((p.distance * 60.0 ) / p.duration ,2) AS speed
		-- Tổng pizza giao
FROM pre_data p
JOIN count_number_of_pizza_in_each_order c ON p.order_id = c.order_id


In [ ]:
%sql
SELECT *
FROM destination.vw_efficiency_for_each_order

customer_id,order_id,number_of_pizzas,runner_id,rating,order_time,pickup_time,distance,preparation_time,duration,speed
101,1,1,1,5,2020-01-01T18:05:02.000Z,2020-01-01T18:15:34.000Z,20.0,10,32,37.5
101,2,1,1,4,2020-01-01T19:00:52.000Z,2020-01-01T19:10:54.000Z,20.0,10,27,44.44
102,3,2,1,3,2020-01-02T23:51:23.000Z,2020-01-03T00:12:37.000Z,13.4,21,20,40.2
103,4,3,2,1,2020-01-04T13:23:46.000Z,2020-01-04T13:53:03.000Z,23.4,29,40,35.1
104,5,1,3,5,2020-01-08T21:00:29.000Z,2020-01-08T21:10:57.000Z,10.0,10,15,40.0
101,6,1,3,null,2020-01-08T21:03:13.000Z,null,null,null,null,null
105,7,1,2,4,2020-01-08T21:20:29.000Z,2020-01-08T21:30:45.000Z,25.0,10,25,60.0
102,8,1,2,5,2020-01-09T23:54:33.000Z,2020-01-10T00:15:02.000Z,23.4,20,15,93.6
103,9,1,2,null,2020-01-10T11:22:59.000Z,null,null,null,null,null
104,10,2,1,5,2020-01-11T18:34:49.000Z,2020-01-11T18:50:20.000Z,10.0,15,10,60.0


In [ ]:
%sql
-- Average speed for each runner
SELECT
		v.runner_id,
		AVG(v.speed) AS avg_speed_for_each_runner
FROM destination.vw_efficiency_for_each_order v
GROUP BY v.runner_id
ORDER BY avg_speed_for_each_runner DESC

runner_id,avg_speed_for_each_runner
2,62.9
1,45.535
3,40.0


**Đọc dữ liệu**

  * D.3: Đã tạo bảng runner_ratings với các cột: rating_id, order_id, rating (1-5 sao), comment, rating_time. Đã chèn 8 bản ghi đánh giá mẫu cho các đơn hàng thành công (order_id 1,2,3,4,5,7,8,10).
  * D.4: Đã tạo view vw_efficiency_for_each_order kết hợp thông tin từ bảng orders, runner_orders, runner_ratings và số lượng pizza, cho thấy các chỉ số: customer_id, order_id, runner_id, rating, order_time, pickup_time, thời gian chuẩn bị (preparation_time), thời gian giao (duration), tốc độ trung bình (speed).
  * Tốc độ trung bình của các runner (tính từ view): runner 2: 62.9 km/h, runner 1: 45.5 km/h, runner 3: 40.0 km/h.
  * Các rating cao nhất (5 sao) thường đi kèm với đơn giao nhanh và thái độ tốt (ví dụ order 1, 5, 8, 10). Rating thấp (1 sao) cho thấy shipper không cẩn thận làm bánh bị xô lệch (order 4).

**Insights**
  
  * Hệ thống rating giúp đo lường trải nghiệm khách hàng một cách rõ ràng và định lượng.
  * Runner 1 có rating ổn định (4-5 sao) nhưng tốc độ trung bình chỉ 45.5 km/h.
  * Runner 2 có tốc độ cao nhất (62.9 km/h) nhưng tốc độ biến động và có rating không cao bằng runner 1. Runner 3 có tốc độ thấp nhất (40 km/h) và chỉ có một đơn đánh giá (rating 3 sao - giao muộn).
  * Có thể kết nối rating với các chỉ số vận hành để cải thiện chất lượng.

**Đề xuất hành động**
  
  * **Triển khai bảng rating vào sản xuất:** Thu thập đánh giá real-time sau mỗi đơn giao thành công, lưu vào bảng runner_ratings. Thiết kế giao diện đơn giản (1-5 sao và ô nhập comment tùy chọn).
  * **Rating Dashboard:** Xây dựng bảng điều khiển theo dõi điểm trung bình (avg rating) theo từng runner và theo khung giờ. Dữ liệu hiện tại cho thấy runner 1 có rating tốt, runner 2 cần cải thiện thái độ, runner 3 cần tăng tốc độ.
  * **Liên kết rating với incentive (khuyến khích):** Runner đạt điểm trung bình từ 4.5 sao trở lên và tỷ lệ giao thành công cao (≥90%) sẽ được thưởng thêm. Điều này khuyến khích runner vừa nhanh vừa cẩn thận.
  * **Sử dụng comment để đào tạo:** Với các đánh giá thấp (ví dụ order 4: "Bánh đến nơi bị xô lệch, shipper không cẩn thận"), cần nhắc nhở runner về cách vận chuyển và xử lý đơn hàng.


### NHÓM 4: TÍNH TOÁN LỢI NHUẬN RÒNG SAU CHI PHÍ VẬN HÀNH

#### D.5.

**If a Meat Lovers pizza was $12 and Vegetarian $10 fixed prices with no cost for extras and each runner is paid $0.30 per kilometre traveled - how much money does Pizza Runner have left over after these deliveries?**


In [ ]:
%sql
;WITH RECURSIVE pre_data AS(
		SELECT
				i.order_id,
				i.customer_orders_id,
				i.pizza_id,
				CASE
					WHEN i.order_id IN (SELECT ro.order_id FROM destination.runner_orders ro WHERE ro.cancellation IS NOT NULL ) THEN 0
					ELSE
						CASE
							WHEN i.pizza_id = 1 THEN 12
							ELSE 10
						END
				END AS money_for_each_pizza
		FROM destination.vw_pizza_ingredient_matrix i
		GROUP BY i.order_id,
				 i.customer_orders_id,
				 i.pizza_id
--		ORDER BY i.order_id ASC,
--				 i.customer_orders_id ASC,
--				 i.pizza_id ASC
), money_for_each_order AS (
		SELECT
				p.order_id,
				SUM(p.money_for_each_pizza) AS money_for_each_order
		FROM	pre_data p
		GROUP BY p.order_id
), ship_fee AS (
		SELECT
			   v.order_id,
			   COALESCE((v.distance * 0.3),0) AS ship_fee
		FROM destination.vw_efficiency_for_each_order v
)
SELECT
		s.order_id,
		CAST( ROUND(m.money_for_each_order,2) AS DECIMAL(10,2) ) AS money_for_each_order,
		CAST( ROUND(s.ship_fee,2) AS DECIMAL(10,2) ) AS ship_fee,
		CAST( ROUND((m.money_for_each_order - s.ship_fee), 2) AS DECIMAL(10,2) ) AS net_profit_for_Danny
FROM money_for_each_order m
JOIN ship_fee s ON m.order_id = s.order_id
ORDER BY s.order_id ASC

order_id,money_for_each_order,ship_fee,net_profit_for_Danny
1,12.00,6.00,6.00
2,12.00,6.00,6.00
3,22.00,4.02,17.98
4,34.00,7.02,26.98
5,12.00,3.00,9.00
6,0.00,0.00,0.00
7,10.00,7.50,2.50
8,12.00,7.02,4.98
9,0.00,0.00,0.00
10,24.00,3.00,21.00


**Đọc dữ liệu**

  * Giả định: giá cố định Meatlovers 12 đô la, Vegetarian 10 đô la, không phí extras, mỗi runner được trả 0.30 đô la trên mỗi km di chuyển. Tính lợi nhuận ròng (doanh thu trừ phí giao hàng) cho từng đơn thành công:

        Order 1: 12 - 6.00 = 6.00 đô la
        Order 2: 12 - 6.00 = 6.00 đô la
        Order 3: 22 - 4.02 = 17.98 đô la
        Order 4: 34 - 7.02 = 26.98 đô la (cao nhất)
        Order 5: 12 - 3.00 = 9.00 đô la
        Order 6: 0 (bị hủy)
        Order 7: 10 - 7.50 = 2.50 đô la (thấp nhất trong các đơn thành công)
        Order 8: 12 - 7.02 = 4.98 đô la
        Order 9: 0 (bị hủy)
        Order 10: 24 - 3.00 = 21.00 đô la

  * Tổng lợi nhuận ròng từ các đơn thành công (không tính đơn hủy) khoảng 94.44 đô la (chưa trừ chi phí nguyên liệu, chỉ trừ phí runner theo km).

**Insights**
  
  * Chi phí runner chiếm tỷ lệ đáng kể trên các đơn có khoảng cách xa (order 3, 4, 7, 8). Ví dụ order 7 có doanh thu 10 đô la nhưng phí giao hàng 7.50 đô la, lợi nhuận chỉ còn 2.50 đô la.
  * Lợi nhuận dương ở hầu hết đơn thành công, nhưng biên lợi nhuận rất mỏng trên một số đơn (order 7 và order 8).
  * Khoảng cách giao hàng (distance) và số lượng pizza (pizza count) ảnh hưởng trực tiếp đến lợi nhuận. Đơn có nhiều pizza và khoảng cách ngắn cho lợi nhuận cao (order 4: 3 pizza, distance 23.4 km nhưng vẫn lãi cao nhờ doanh thu lớn 34 đô la).

**Đề xuất hành động**
  
  * **Dynamic pricing theo khoảng cách (định giá linh hoạt):** Thu thêm phí giao hàng cho các đơn có distance > 15 km, ví dụ cộng thêm 1-2 đô la. Dữ liệu cho thấy order 7 (distance 25 km) lợi nhuận rất thấp (2.50 đô la), cần điều chỉnh để không bị lỗ khi chi phí vận hành tăng.
  * **Tối ưu tuyến đường và ghép đơn:** Sử dụng kỹ thuật tối ưu hóa lộ trình (route optimization) để giảm km di chuyển rỗng và ghép nhiều đơn trên cùng một chuyến. Dữ liệu về khoảng cách trung bình mỗi khách (từ File 02, B.4) cho thấy khách 103 và 105 ở xa, có thể ưu tiên một runner phụ trách khu vực đó để giảm tổng km.
  * **Ngưỡng lợi nhuận tối thiểu (profit threshold):** Không chấp nhận các đơn dự kiến lợi nhuận dưới 5 đô la sau khi trừ phí runner (trừ khi khách hàng là VIP hoặc order nhiều pizza). Dữ liệu order 7 chỉ lãi 2.50 đô la, cần xem xét tăng giá hoặc từ chối nếu khoảng cách quá xa.
  * **Theo dõi lợi nhuận ròng hàng tuần:** Xây dựng báo cáo dựa trên view vw_efficiency_for_each_order và bảng giá vốn nguyên liệu (từ File 03) để tính lợi nhuận thực tế (doanh thu - phí runner - chi phí nguyên liệu). Điều này giúp ra quyết định điều chỉnh giá bán hoặc chiến lược giao hàng.
  


## III. TỔNG KẾT

**Liên kết các insights từ các nhóm**

  * •	Nhóm 1 (định giá cơ bản) cho thấy doanh thu hiện tại 138 đô la, chủ yếu từ Meatlovers. Điều này phù hợp với phân tích từ File 03 (Meatlovers có lượng nguyên liệu tiêu thụ cao, đặc biệt Bacon và Cheese). Nếu tăng giá Meatlovers lên 13 đô la, cần xem xét phản ứng khách hàng
  * •	Nhóm 1 (định giá cơ bản) cho thấy doanh thu hiện tại 138 đô la, chủ yếu từ Meatlovers. Điều này phù hợp với phân tích từ File 03 (Meatlovers có lượng nguyên liệu tiêu thụ cao, đặc biệt Bacon và Cheese). Nếu tăng giá Meatlovers lên 13 đô la, cần xem xét phản ứng khách hàng
  * •	Nhóm 3 (rating) cho thấy runner 1 có rating ổn định và tốc độ vừa phải (45.5 km/h), runner 2 nhanh (62.9 km/h) nhưng có thể không cẩn thận (order 4 bị rating 1 sao). Runner 3 chậm (40 km/h) và rating trung bình (3 sao). Liên kết với File 02 (tỷ lệ thành công sau khi loại trừ hủy khách quan là 100% cả ba) nhưng chất lượng cảm nhận khác biệt. Do đó, cần kết hợp rating vào incentive để cải thiện trải nghiệm.
  * •	Nhóm 4 (lợi nhuận ròng) cho thấy các đơn có khoảng cách xa (order 7: distance 25 km) có lợi nhuận rất mỏng (2.50 đô la). Kết hợp với File 02 (khoảng cách trung bình khách 105 là 25 km), có thể thấy những khách ở xa cần phụ thu phí giao hàng hoặc khuyến khích đặt với số lượng lớn để bù đắp chi phí.


**Đề xuất tổng hợp**


  * Định giá và phí extras: Giữ giá cơ bản 12/10 đô la cho pizza. Áp dụng phí 1 đô la cho mỗi extra topping (đặc biệt Bacon). Đối với Cheese, chuyển từ topping mặc định thành tùy chọn "Extra Cheese" (có phí) để vừa giảm lãng phí vừa tăng doanh thu.
  * Thu phí giao hàng theo khoảng cách: Với khách hàng có distance > 15 km (như khách 103, 105), cộng thêm 1-2 đô la phí giao xa. Dữ liệu từ D.5 cho thấy order 7 và 8 có lợi nhuận rất thấp, phí này sẽ cải thiện biên lợi nhuận.
  * Tích hợp rating vào incentive: Runner đạt rating trung bình >= 4.5 sao và tốc độ trong ngưỡng an toàn (< 60 km/h) được thưởng thêm 5-10% trên số đơn. Điều này khuyến khích runner vừa nhanh vừa cẩn thận (như runner 1).
  * Theo dõi lợi nhuận ròng hàng tuần: Xây dựng báo cáo tổng hợp doanh thu (D.2), phí runner (D.5), và chi phí nguyên liệu (từ File 03) để có lợi nhuận thực tế. Sử dụng view vw_efficiency_for_each_order để phân tích chi tiết theo từng đơn.

**Ưu tiên thực hiện**


  * Triển khai thu phí extras 1 đô la và cập nhật giao diện app.
  * Hoàn thiện hệ thống rating (bảng runner_ratings) và dashboard đánh giá.
  * Áp dụng phí giao xa (distance surcharge) cho các đơn trên 15 km.


## PHỤ LỤC: KHÁC BIỆT CÚ PHÁP GIỮA T‑SQL VÀ DATABRICKS SQL

| Vấn đề | T‑SQL (SQL Server) | Databricks SQL (Spark SQL) | Lý do |
|--------|-------------------|----------------------------|-------|
| **Tự tăng ID** | `INT IDENTITY(1,1) PRIMARY KEY` | `BIGINT GENERATED ALWAYS AS IDENTITY` | Spark không hỗ trợ cú pháp `IDENTITY(seed, increment)`; dùng `GENERATED ALWAYS AS IDENTITY`. |
| **Kiểu chuỗi Unicode** | `NVARCHAR(255)` | `STRING` | Databricks không phân biệt VARCHAR/NVARCHAR; dùng `STRING` cho mọi văn bản. |
| **Khóa ngoại trong cột** | `order_id INT NOT NULL FOREIGN KEY REFERENCES ...` | Phải tách riêng: `order_id INT NOT NULL, FOREIGN KEY (order_id) REFERENCES ...` | Spark không hỗ trợ khai báo `FOREIGN KEY` trực tiếp trong định nghĩa cột. |
| **Giá trị mặc định thời gian** | `DATETIME DEFAULT GETDATE()` | `TIMESTAMP DEFAULT CURRENT_TIMESTAMP()` | `GETDATE()` không có; dùng `CURRENT_TIMESTAMP()`. Kiểu `DATETIME` cũng đổi thành `TIMESTAMP`. |
| **Chuỗi Unicode** | `N'Tiếng Việt có dấu'` | `'Tiếng Việt có dấu'` (không cần `N`) | Databricks mặc định hỗ trợ UTF‑8, không cần tiền tố `N` như SQL Server. |
| **Hàm thay thế NULL** | `ISNULL((v.distance * 0.3), 0)` | `COALESCE((v.distance * 0.3), 0)` | `ISNULL` là hàm riêng của T‑SQL; Spark dùng `COALESCE` theo chuẩn ANSI. |
| **CTE đệ quy** | `WITH pre_data AS (...)` | `WITH RECURSIVE pre_data AS (...)` | Spark yêu cầu từ khóa `RECURSIVE` cho CTE tự tham chiếu. |
| **Kết quả số thập phân cố định** | `ROUND(expr, 2)` có thể trả về `DECIMAL` tuỳ biểu thức | `CAST(ROUND(expr, 2) AS DECIMAL(10,2))` bắt buộc nếu muốn giữ định dạng tiền tệ | `ROUND` trong Spark chỉ làm tròn, không thay đổi kiểu dữ liệu (vẫn là DOUBLE). |